# Python data-tools practice (Module A)

Hands-on practice for the four libraries from Module A: **NumPy**, **pandas**, **matplotlib**, and **xarray**.

## How to use this notebook

For each exercise:

1. Read the task.
2. **Use the empty cell above each solution to try it yourself first.** That is where the learning happens, and it is fine to get it wrong a few times.
3. Then run the solution cell and read it. Each one ends with an `assert`, a self-check that passes silently when the answer is right and complains loudly when it is not. That is the same "give the work a verifier" idea from Module B, applied to your own practice.

Nothing here is graded. Work through what is useful, skip what is easy, and bring anything that stays confusing to office hours.

Run this setup cell first.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr

# the course datasets (same files used in the lectures)
taxi = pd.read_csv("../data/2019_Yellow_Taxi_Trip_Data.csv")
temps = xr.open_dataset("../data/global_temparature.nc")

print("taxi:", taxi.shape[0], "rows,", taxi.shape[1], "columns")
print("temps:", dict(temps.sizes))

# 1. NumPy

A quick refresher, then five exercises. Remember: the two things that trip people up are **shapes** (what dimensions does the result have?) and **axes** (which direction does a reduction go?).

### Refresher

`np.arange`, reshaping, boolean masks, and reducing along an axis.

In [ ]:
a = np.arange(12)              # [0, 1, 2, ..., 11]
m = a.reshape(3, 4)            # a 3-by-4 matrix
print("m =\n", m)
print("even entries:", a[a % 2 == 0])     # a boolean mask selects elements
print("column sums (axis=0):", m.sum(axis=0))   # collapse the rows, one sum per column

### Exercise 1.1 - reshape and index

Make a 1-D array of the integers `0` to `11`, reshape it into a 4-by-3 matrix called `grid`, and read out the element in row 2, column 1 as `value`.

In [ ]:
# Your turn. Try it here, then run the solution cell below to check.

In [ ]:
# --- solution ---
grid = np.arange(12).reshape(4, 3)
value = grid[2, 1]
print(grid, "\nvalue =", value)

assert grid.shape == (4, 3)
assert value == 7        # row 2, col 1: rows 0,1,2 -> 6,7,8 start; [2,1] = 7
print("check passed")

### Exercise 1.2 - boolean mask

Given `x = np.array([3, 8, 1, 9, 4, 7, 2])`, build `big`, the elements greater than 4, and `how_many`, their count.

In [ ]:
# Your turn. Try it here, then run the solution cell below to check.

In [ ]:
# --- solution ---
x = np.array([3, 8, 1, 9, 4, 7, 2])
big = x[x > 4]
how_many = (x > 4).sum()
print("big =", big, "| how_many =", how_many)

assert list(big) == [8, 9, 7]
assert how_many == 3
print("check passed")

### Exercise 1.3 - broadcasting

Without writing a loop, make a 3-by-4 multiplication table `table` where `table[i, j] == i * j`. (Hint: multiply a column vector `0,1,2` by a row vector `0,1,2,3`.)

In [ ]:
# Your turn. Try it here, then run the solution cell below to check.

In [ ]:
# --- solution ---
col = np.arange(3).reshape(3, 1)     # shape (3, 1)
row = np.arange(4).reshape(1, 4)     # shape (1, 4)
table = col * row                    # broadcasts to (3, 4)
print(table)

assert table.shape == (3, 4)
assert table[2, 3] == 6
print("check passed")

### Exercise 1.4 - reducing along an axis

Given the matrix `M` below, compute `col_means` (the mean of each column) and `row_maxes` (the largest value in each row). Watch the axis.

In [ ]:
# Your turn. Try it here, then run the solution cell below to check.

In [ ]:
# --- solution ---
M = np.array([[1.0, 2, 3, 4],
              [5, 6, 7, 8],
              [9, 10, 11, 12]])
col_means = M.mean(axis=0)      # collapse rows -> one value per column -> shape (4,)
row_maxes = M.max(axis=1)       # collapse columns -> one value per row -> shape (3,)
print("col_means =", col_means)
print("row_maxes =", row_maxes)

assert col_means.shape == (4,) and list(col_means) == [5, 6, 7, 8]
assert row_maxes.shape == (3,) and list(row_maxes) == [4, 8, 12]
print("check passed")

### Exercise 1.5 - find and fix the bug

The code below is meant to normalise each **row** so it sums to 1 (useful for turning counts into proportions). It runs without error but gives the wrong answer. Fix it so `norm` has each row summing to 1.

```python
counts = np.array([[1, 1, 2],
                   [2, 2, 4]], dtype=float)
norm = counts / counts.sum(axis=0)     # bug is on this line
```

In [ ]:
# Your turn. Try it here, then run the solution cell below to check.

In [ ]:
# --- solution ---
counts = np.array([[1, 1, 2],
                   [2, 2, 4]], dtype=float)
# The bug: axis=0 sums down the columns, so each row is divided by the wrong totals.
# To normalise rows we need the per-row sum, and it must keep its shape to broadcast back.
norm = counts / counts.sum(axis=1, keepdims=True)
print(norm)

assert np.allclose(norm.sum(axis=1), 1.0)
print("check passed")

# 2. pandas

Using the taxi sample (`taxi`). Reminder of the columns you will need: `payment_type` (1 = card, 2 = cash), `trip_distance`, `fare_amount`, `tip_amount`, `total_amount`, `pulocationid` (pickup zone), `tpep_pickup_datetime`.

### Refresher

Filtering with a boolean mask, adding a column, and `group_by`.

In [ ]:
card = taxi[taxi["payment_type"] == 1]          # filter rows
print("card trips:", len(card))
print("mean fare per zone (first 3):")
print(taxi.groupby("pulocationid")["fare_amount"].mean().head(3))

### Exercise 2.1 - filter and count

How many trips were paid in **cash** (`payment_type == 2`)? Put the count in `n_cash`.

In [ ]:
# Your turn. Try it here, then run the solution cell below to check.

In [ ]:
# --- solution ---
n_cash = (taxi["payment_type"] == 2).sum()     # or: len(taxi[taxi["payment_type"] == 2])
print("cash trips:", n_cash)

assert n_cash == 2803
print("check passed")

### Exercise 2.2 - a new column

Add a column `tip_pct` giving the tip as a percentage of `total_amount`. Then find the mean `tip_pct` over **card** trips only (`payment_type == 1`), as `mean_card_tip`. Only consider rows with a positive `total_amount` so the percentage is meaningful.

In [ ]:
# Your turn. Try it here, then run the solution cell below to check.

In [ ]:
# --- solution ---
clean = taxi[taxi["total_amount"] > 0].copy()
clean["tip_pct"] = clean["tip_amount"] / clean["total_amount"] * 100
mean_card_tip = clean.loc[clean["payment_type"] == 1, "tip_pct"].mean()
print(f"mean tip %, card trips: {mean_card_tip:.1f}")

assert 0 < mean_card_tip < 30     # a sane tip percentage
print("check passed")

### Exercise 2.3 - group-by

Compute the number of trips starting in each pickup zone, as a Series `trips_per_zone` indexed by `pulocationid`, sorted from busiest to quietest. What is the busiest zone's id (`busiest`)?

In [ ]:
# Your turn. Try it here, then run the solution cell below to check.

In [ ]:
# --- solution ---
trips_per_zone = taxi.groupby("pulocationid").size().sort_values(ascending=False)
busiest = trips_per_zone.index[0]
print("busiest zone:", busiest, "with", trips_per_zone.iloc[0], "trips")
print(trips_per_zone.head())

assert trips_per_zone.sum() == len(taxi)      # every trip counted once
assert trips_per_zone.iloc[0] >= trips_per_zone.iloc[-1]
print("check passed")

### Exercise 2.4 - parse a datetime

`tpep_pickup_datetime` is stored as text. Convert it to a real datetime, extract the hour of day, and build `trips_by_hour`, the number of pickups in each hour (0 to 23).

In [ ]:
# Your turn. Try it here, then run the solution cell below to check.

In [ ]:
# --- solution ---
pickup = pd.to_datetime(taxi["tpep_pickup_datetime"])
trips_by_hour = pickup.dt.hour.value_counts().sort_index()
print(trips_by_hour)

assert trips_by_hour.sum() == len(taxi)
assert trips_by_hour.index.min() >= 0 and trips_by_hour.index.max() <= 23
print("check passed")

### Exercise 2.5 - the many-to-many join trap

A join pairs **every** matching row on the left with **every** matching row on the right. If a key repeats on both sides, the row count can explode. Predict the number of rows in the merge below, then check.

In [ ]:
# Your turn. Try it here, then run the solution cell below to check.

In [ ]:
# --- solution ---
left = pd.DataFrame({"zone": [1, 1, 2], "trips": [10, 20, 30]})
right = pd.DataFrame({"zone": [1, 1, 2], "name": ["A", "B", "C"]})

merged = pd.merge(left, right, on="zone")
print(merged)
# zone 1 matches 2 x 2 = 4 ways; zone 2 matches 1 x 1 = 1 way; total 5
print("rows:", len(merged))

assert len(merged) == 5
# the guard: declaring what you expect turns a silent blow-up into an error
try:
    pd.merge(left, right, on="zone", validate="one_to_one")
    raise RuntimeError("should have failed")
except pd.errors.MergeError:
    print("validate='one_to_one' correctly rejected the duplicate keys")
print("check passed")

### Exercise 2.6 - find and fix the silent bug

Someone computes the average tip percentage per zone like this, and concludes most zones barely tip. The code runs fine. What makes the numbers misleading, and how do you fix it? (Hint: think about who records a tip of 0.)

```python
taxi["tip_pct"] = taxi["tip_amount"] / taxi["total_amount"] * 100
by_zone = taxi.groupby("pulocationid")["tip_pct"].mean()
```

In [ ]:
# Your turn. Try it here, then run the solution cell below to check.

In [ ]:
# --- solution ---
# Cash trips (payment_type == 2) record tip_amount as 0, because the driver is tipped
# in cash off the books. Averaging them in drags every zone's tip % down. Card trips
# are the ones where the tip is actually recorded, so filter to those first.
cards = taxi[(taxi["payment_type"] == 1) & (taxi["total_amount"] > 0)].copy()
cards["tip_pct"] = cards["tip_amount"] / cards["total_amount"] * 100
by_zone_card = cards.groupby("pulocationid")["tip_pct"].mean()

all_trips = taxi[taxi["total_amount"] > 0].copy()
all_trips["tip_pct"] = all_trips["tip_amount"] / all_trips["total_amount"] * 100
by_zone_all = all_trips.groupby("pulocationid")["tip_pct"].mean()

print(f"mean over zones, all trips:  {by_zone_all.mean():.1f}%")
print(f"mean over zones, card only:  {by_zone_card.mean():.1f}%")

assert by_zone_card.mean() > by_zone_all.mean()    # card-only is meaningfully higher
print("check passed: the cash-zero trips were hiding the real tipping rate")

# 3. matplotlib

Three exercises. The self-checks here look at the figure you built (its labels, how many lines or bars it has) rather than a single number, because that is what a plot is. Use the **figure and axes** model: `fig, ax = plt.subplots()`, then draw on `ax`.

### Refresher

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
x = np.linspace(0, 10, 100)
ax.plot(x, x ** 2, label="y = x^2")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()

### Exercise 3.1 - a labelled line plot

Plot `y = sqrt(x)` for `x` from 0 to 25 on an axes called `ax1`. Give it an x-label and a y-label. (The check looks for one line and both labels.)

In [ ]:
# Your turn. Try it here, then run the solution cell below to check.

In [ ]:
# --- solution ---
fig, ax1 = plt.subplots(figsize=(5, 3))
x = np.linspace(0, 25, 100)
ax1.plot(x, np.sqrt(x))
ax1.set_xlabel("x")
ax1.set_ylabel("sqrt(x)")

assert len(ax1.lines) == 1
assert ax1.get_xlabel() != "" and ax1.get_ylabel() != ""
print("check passed")

### Exercise 3.2 - a histogram from data

Draw a histogram of `trip_distance` for trips shorter than 20 miles, on an axes `ax2`, using 30 bins. Label the x-axis "trip distance (miles)".

In [ ]:
# Your turn. Try it here, then run the solution cell below to check.

In [ ]:
# --- solution ---
short = taxi.loc[taxi["trip_distance"] < 20, "trip_distance"]
fig, ax2 = plt.subplots(figsize=(5, 3))
ax2.hist(short, bins=30)
ax2.set_xlabel("trip distance (miles)")
ax2.set_ylabel("number of trips")

assert len(ax2.patches) == 30            # 30 bins -> 30 bars
assert ax2.get_xlabel() == "trip distance (miles)"
print("check passed")

### Exercise 3.3 - two panels side by side

Make a figure with **two** axes side by side (`fig, axes = plt.subplots(1, 2, ...)`). On the left, plot the sorted `fare_amount` values; on the right, a histogram of `passenger_count`. The check looks for two axes.

In [ ]:
# Your turn. Try it here, then run the solution cell below to check.

In [ ]:
# --- solution ---
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].plot(np.sort(taxi["fare_amount"].values))
axes[0].set_title("fares, sorted")
axes[1].hist(taxi["passenger_count"], bins=range(0, 8))
axes[1].set_title("passenger count")

assert axes.shape == (2,)
assert len(axes[0].lines) == 1
print("check passed")

# 4. xarray

Using `temps`, the global surface temperature dataset. It has one variable, `tas`, with dimensions `(time, lat, lon)`. xarray is like pandas for labelled arrays with more than two dimensions: you select and reduce by **dimension name**, not by position.

### Refresher

Select by label with `.sel`, reduce along named dimensions with `.mean`.

In [ ]:
print("dims:", dict(temps.sizes))
one_year = temps["tas"].sel(time=2000)        # select by the label 2000, not position
print("one year's map has shape:", one_year.shape)     # (lat, lon)
global_each_year = temps["tas"].mean(["lat", "lon"])   # collapse space, keep time
print("global-mean series length:", global_each_year.sizes["time"])

### Exercise 4.1 - select by label

Pull out the temperature map for the year **2050** as `map_2050`, and confirm it is a 2-D field over `(lat, lon)`.

In [ ]:
# Your turn. Try it here, then run the solution cell below to check.

In [ ]:
# --- solution ---
map_2050 = temps["tas"].sel(time=2050)
print("shape:", map_2050.shape, "dims:", map_2050.dims)

assert map_2050.dims == ("lat", "lon")
assert map_2050.sizes["lat"] == 96 and map_2050.sizes["lon"] == 144
print("check passed")

### Exercise 4.2 - reduce along named dimensions

Build `global_mean`, the area-*unweighted* global mean temperature for every year (collapse `lat` and `lon`, keep `time`). It should be a 1-D series over `time`.

In [ ]:
# Your turn. Try it here, then run the solution cell below to check.

In [ ]:
# --- solution ---
global_mean = temps["tas"].mean(["lat", "lon"])
print("length:", global_mean.sizes["time"], "| dims:", global_mean.dims)
print("first and last year:", float(global_mean.isel(time=0)), "->", float(global_mean.isel(time=-1)))

assert global_mean.dims == ("time",)
assert global_mean.sizes["time"] == 250
print("check passed")

### Exercise 4.3 - a weighted mean (and why it matters)

A plain mean over `lat` and `lon` is wrong for a globe, because grid cells near the poles cover far less area than cells near the equator. The fix is to weight each latitude by `cos(latitude)`.

Build `weights = cos(latitude in radians)`, then use xarray's `.weighted(...).mean(...)` to get `weighted_global`. Check that it differs from the unweighted `global_mean`.

In [ ]:
# Your turn. Try it here, then run the solution cell below to check.

In [ ]:
# --- solution ---
weights = np.cos(np.deg2rad(temps["lat"]))
weighted_global = temps["tas"].weighted(weights).mean(["lat", "lon"])
print("unweighted 2050:", float(global_mean.sel(time=2050)))
print("weighted   2050:", float(weighted_global.sel(time=2050)))

assert weighted_global.dims == ("time",)
assert not np.isclose(float(weighted_global.sel(time=2050)),
                      float(global_mean.sel(time=2050)))   # weighting changes the answer
print("check passed")

### Exercise 4.4 - broadcasting by name

xarray lines arrays up by dimension name, so you do not have to worry about axis order. Multiply the year-2000 map (`tas` at `time=2000`, dims `(lat, lon)`) by the 1-D `weights` (dim `lat`) and confirm the result spreads back over `(lat, lon)`.

In [ ]:
# Your turn. Try it here, then run the solution cell below to check.

In [ ]:
# --- solution ---
scaled = temps["tas"].sel(time=2000) * weights      # (lat, lon) * (lat,) -> (lat, lon)
print("result dims:", scaled.dims, "shape:", scaled.shape)

assert scaled.dims == ("lat", "lon")
print("check passed")

# Done

If the hard ones were the pandas group-by and the xarray weighted mean, you are in good company; those are the two that matter most for the project. Re-run any exercise with your own variation (a different zone, a different year) to be sure you have really got it.

Next places to practise:
- `practice/working_with_agents.md` for hands-on agent practice.
- `practice/skills_map.md` for the full list of skills the course is building.
- The Module A lecture notebooks, for worked examples of everything above.